# 04 · Descubrimiento y verificación de la web oficial

**Componente B · Bloque 2 del Company Profile** · Capa: **plata** (`plata/urls_piloto.csv`)

La SCVS no publica la web de cada empresa. Este notebook propone dominios candidatos y los verifica **sin buscadores** (sus términos prohíben la consulta automatizada):

```mermaid
flowchart LR
    P[piloto_2025.csv] --> N[razón social limpia<br/>+ nombre comercial]
    N --> D[dominios candidatos<br/>.com.ec · .ec · .com]
    P -->|J61| PDB[PeeringDB API]
    PDB --> D
    D --> DNS{¿resuelve<br/>en DNS?}
    DNS -- sí --> S[Scrapling ético:<br/>robots.txt · 1 req/dominio · 3 s]
    S --> V{¿RUC o razón social<br/>en la portada ≥ 85?}
    V -- sí --> OK[url_verificada = si]
    V -- no --> R[revisar a mano]
```

**Reglas éticas** ([`investigacion-b/06`](../../docs/investigacion-b/06-scraping-etico.md)): respeta robots.txt, bot identificado, sin suplantar navegador, una petición por dominio cada 3 s, sin reintentos ante bloqueos, solo la portada.

In [1]:
from pathlib import Path

ETL = Path.cwd().resolve()
if ETL.name == "notebooks":
    ETL = ETL.parent
DATOS = ETL / "datos"
CRUDO = DATOS / "crudo"      # bronce: datos tal como llegan, sin modificar
SCVS = CRUDO / "2025"        # descargas oficiales de la SCVS
WEB = CRUDO / "web"          # páginas capturadas de las webs oficiales
PLATA = DATOS / "plata"      # datos limpios y validados
ORO = DATOS / "oro"          # datos listos para publicar
for carpeta in (WEB, PLATA, ORO):
    carpeta.mkdir(parents=True, exist_ok=True)

In [ ]:
import datetime as dt
import logging
import re
import socket
import time
import unicodedata
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import urlparse

import pandas as pd
from rapidfuzz import fuzz
from scrapling.fetchers import Fetcher, FetcherSession
from scrapling.spiders import Request, Spider

USER_AGENT = "ObservatorioEPN-Bot/0.1 (+https://github.com/alda244/observatorio-epn; proyecto academico EPN)"
UMBRAL = 85
LIMITE_POR_CRITERIO = 2   # empresas por criterio del piloto en esta corrida; None = las 500

## 1. Empresas a buscar

In [3]:
piloto = pd.read_csv(ORO / "piloto_2025.csv", dtype={"ruc": str, "expediente": int})
SALIDA = PLATA / "urls_piloto.csv"
previo = pd.read_csv(SALIDA, dtype={"expediente": int}) if SALIDA.exists() else pd.DataFrame(columns=["expediente", "url_verificada"])

# No se repite la búsqueda de las ya verificadas ni de las corregidas a mano.
cerradas = set(previo.loc[previo["url_verificada"].isin(["si", "manual"]), "expediente"])
pendientes = piloto[~piloto["expediente"].isin(cerradas)]
if LIMITE_POR_CRITERIO:
    pendientes = pendientes.groupby("criterio_piloto").head(LIMITE_POR_CRITERIO)
pendientes[["expediente", "nombre", "grupo", "ciiu", "ciudad"]]

,expediente,nombre,grupo,ciiu,ciudad
0,47845,CONSORCIO ECUATORIANO DE TELECOMUNICACIONES S....,TEC,J6120.01,GUAYAQUIL
1,47972,OTECEL S.A.,TEC,J6120.01,QUITO
301,2913,CORPORACION EL ROSADO S.A.,G,G4711.01,GUAYAQUIL
302,86945,PRIMAX COMERCIAL DEL ECUADOR SOCIEDAD ANONIMA,G,G4661.03,QUITO
351,2151,SOCIEDAD NACIONAL DE GALAPAGOS CA,C,C1020.01,GUAYAQUIL
352,742,LA FABRIL S.A.,C,C1040.13,MONTECRISTI
402,740608,"MCKINSEY & COMPANY, INC. ECUADOR",M,M7020.04,QUITO
403,704429,IMS-INTERNET MEDIA SERVICES ECUADOR S.A.,M,M7310.01,QUITO
450,48038,SALUDSA SISTEMA DE MEDICINA PRE-PAGADA DEL ECU...,K,K6512.02,QUITO
451,151460,BMI IGUALAS MEDICAS DEL ECUADOR S.A.,K,K6512.02,QUITO


## 2. Nombre limpio y nombre comercial

Muchas razones sociales terminan con el nombre comercial después de la forma jurídica: *"CONSORCIO ECUATORIANO DE TELECOMUNICACIONES S.A. **CONECEL**"*.

In [4]:
FORMAS = r"\b(S\.?\s?A\.?\s?S\.?|S\.?\s?A\.?|C\.?\s?A\.?|C\.?\s?L\.?|C[IÍ]A\.?\s?LTDA\.?|LTDA\.?|SOCIEDAD AN[OÓ]NIMA|COMPA[ÑN][IÍ]A LIMITADA|SOCIEDAD POR ACCIONES SIMPLIFICADA|SOCIEDAD DE BENEFICIO E INTER[EÉ]S COLECTIVO)\b"
GENERICAS = {"SOCIEDAD", "CORPORACION", "CONSORCIO", "COMPANIA", "GRUPO", "NACIONAL", "INDUSTRIAL", "INDUSTRIAS",
             "PROCESADORA", "DISTRIBUIDORA", "COMERCIAL", "SERVICIOS", "SISTEMA", "TIENDAS", "EMPRESA"}
VACIAS = {"DE", "DEL", "LA", "LAS", "LOS", "EL", "Y", "E", "EN", "PARA", "CIA", "CIA.", "COMPANIA", "ECUADOR", "ECUATORIANA", "ECUATORIANO"}

def sin_tildes(texto: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFD", texto) if unicodedata.category(c) != "Mn")

def normalizar(texto: str) -> str:
    return " ".join(re.sub(r"[^A-Z0-9 ]+", " ", sin_tildes(str(texto)).upper()).split())

def partes_nombre(razon_social: str) -> tuple[str, str]:
    """Devuelve (razón social sin forma jurídica, nombre comercial que sigue a la forma jurídica)."""
    texto = sin_tildes(razon_social).upper()
    m = list(re.finditer(FORMAS, texto))
    comercial = normalizar(texto[m[-1].end():]) if m else ""
    base = normalizar(re.sub(FORMAS, " ", texto[: m[-1].start()] if m and comercial else texto))
    return base, comercial

pd.DataFrame([(n, *partes_nombre(n)) for n in pendientes["nombre"]], columns=["razón social", "base", "comercial"])

,razón social,base,comercial
0,CONSORCIO ECUATORIANO DE TELECOMUNICACIONES S....,CONSORCIO ECUATORIANO DE TELECOMUNICACIONES,CONECEL
1,OTECEL S.A.,OTECEL,
2,CORPORACION EL ROSADO S.A.,CORPORACION EL ROSADO,
3,PRIMAX COMERCIAL DEL ECUADOR SOCIEDAD ANONIMA,PRIMAX COMERCIAL DEL ECUADOR,
4,SOCIEDAD NACIONAL DE GALAPAGOS CA,SOCIEDAD NACIONAL DE GALAPAGOS,
5,LA FABRIL S.A.,LA FABRIL,
6,"MCKINSEY & COMPANY, INC. ECUADOR",MCKINSEY COMPANY INC ECUADOR,
7,IMS-INTERNET MEDIA SERVICES ECUADOR S.A.,IMS INTERNET MEDIA SERVICES ECUADOR,
8,SALUDSA SISTEMA DE MEDICINA PRE-PAGADA DEL ECU...,SALUDSA SISTEMA DE MEDICINA PRE PAGADA DEL ECU...,
9,BMI IGUALAS MEDICAS DEL ECUADOR S.A.,BMI IGUALAS MEDICAS DEL ECUADOR,


## 3. Dominios candidatos (sin tocar ninguna web todavía)

In [5]:
TLDS = [".com.ec", ".ec", ".com"]

def slugs(base: str, comercial: str) -> list[str]:
    tokens = [t for t in base.split() if t not in VACIAS and len(t) > 1]
    opciones = []
    if comercial and len(comercial.replace(" ", "")) >= 4:
        opciones.append(comercial.replace(" ", "").lower())
    if tokens:
        if tokens[0] not in GENERICAS:   # "sociedad.com" o "corporacion.com" no identifican a nadie
            opciones.append(tokens[0].lower())
        opciones.append("".join(tokens[:2]).lower())
        if len(tokens) <= 3:
            opciones.append("".join(tokens).lower())
    vistos, salida = set(), []
    for s in opciones:
        if len(s) >= 4 and s not in vistos:
            vistos.add(s); salida.append(s)
    return salida[:4]

filas = []
for e in pendientes.itertuples():
    base, comercial = partes_nombre(e.nombre)
    for s in slugs(base, comercial):
        for tld in TLDS:
            filas.append({"expediente": e.expediente, "dominio": s + tld, "origen": "dominio_derivado"})
candidatos = pd.DataFrame(filas)
len(candidatos)

51

### PeeringDB para telecomunicaciones (J61)

[PeeringDB](https://www.peeringdb.com/apidocs/) es un registro público de redes con API abierta las operadoras publican ahí su web. Se consulta con pausa de 4 s entre llamadas.

In [6]:
def web_peeringdb(nombre: str) -> str | None:
    base, comercial = partes_nombre(nombre)
    termino = comercial or base.split()[0]
    r = Fetcher.get(
        "https://www.peeringdb.com/api/net",
        params={"name__contains": termino, "fields": "name,website"},
        headers={"User-Agent": USER_AGENT}, impersonate=None, stealthy_headers=False, timeout=20,
    )
    time.sleep(4)
    if r.status != 200:
        return None
    for red in r.json().get("data", []):
        if red.get("website"):
            return urlparse(red["website"]).netloc.removeprefix("www.")
    return None

j61 = pendientes[pendientes["ciiu"].str.startswith("J61")]
extra = []
for e in j61.itertuples():
    dominio = web_peeringdb(e.nombre)
    if dominio:
        extra.append({"expediente": e.expediente, "dominio": dominio, "origen": "peeringdb"})
candidatos = pd.concat([candidatos, pd.DataFrame(extra)], ignore_index=True).drop_duplicates(["expediente", "dominio"])
pd.DataFrame(extra)

[2026-11-11 06:07:36] INFO: Fetched (200) <GET https://www.peeringdb.com/api/net?name__contains=CONECEL&fields=name%2Cwebsite> (referer: None)


[2026-11-11 06:07:41] INFO: Fetched (200) <GET https://www.peeringdb.com/api/net?name__contains=OTECEL&fields=name%2Cwebsite> (referer: None)


,expediente,dominio,origen
0,47845,claro.com.ec,peeringdb
1,47972,tigo.com.ec,peeringdb


## 4. Filtro DNS: solo dominios que existen

In [7]:
def resuelve(dominio: str) -> bool:
    try:
        socket.getaddrinfo(dominio, 443)
        return True
    except OSError:
        return False

with ThreadPoolExecutor(max_workers=16) as pool:
    candidatos["resuelve"] = list(pool.map(resuelve, candidatos["dominio"]))
vivos = candidatos[candidatos["resuelve"]].reset_index(drop=True)
print(f"{len(vivos)} de {len(candidatos)} dominios existen")

13 de 53 dominios existen


## 5. Visita ética de la portada con Scrapling

In [8]:
class VerificadorPortada(Spider):
    name = "verificador_portada"
    robots_txt_obey = True                 # respeta robots.txt
    concurrent_requests = 4
    concurrent_requests_per_domain = 1     # nunca dos peticiones simultáneas al mismo sitio
    download_delay = 3.0                   # 3 s entre peticiones al mismo sitio
    max_blocked_retries = 0                # si el sitio bloquea, no se insiste
    logging_level = logging.WARNING

    def __init__(self, candidatos: pd.DataFrame):
        self.candidatos = candidatos
        super().__init__()

    def configure_sessions(self, manager):
        manager.add("etico", FetcherSession(
            impersonate=None, stealthy_headers=False,   # sin suplantar un navegador
            headers={"User-Agent": USER_AGENT}, retries=1, timeout=20,
        ))

    async def start_requests(self):
        for c in self.candidatos.itertuples():
            yield Request(f"https://{c.dominio}/", sid="etico", callback=self.parse, dont_filter=True,
                          meta={"expediente": c.expediente, "dominio": c.dominio, "origen": c.origen})

    async def parse(self, r):
        texto = " ".join(r.css("body *::text").getall())
        yield {
            **{k: r.meta[k] for k in ("expediente", "dominio", "origen")},
            "status": r.status,
            "url_final": r.url,
            "title": r.css("title::text").get() or "",
            "og_site_name": r.css('meta[property="og:site_name"]::attr(content)').get() or "",
            "texto": texto[:30_000],
        }

inicio = time.time()
visitas = [item async for item in VerificadorPortada(vivos).stream()]
print(f"{len(visitas)} portadas leídas de {len(vivos)} dominios en {time.time() - inicio:.0f} s")

[2026-11-11 06:07:56]:(verificador_portada) WARNING: Max retries exceeded for blocked request: https://conecel.com/


[2026-11-11 06:08:00]:(verificador_portada) ERROR: Failed after 1 attempts: Failed to perform, curl: (92) HTTP/2 stream 1 reset by server (error 0x2 INTERNAL_ERROR). See https://curl.se/libcurl/c/libcurl-errors.html first for more details.


[2026-11-11 06:08:00]:(verificador_portada) WARNING: Failed to fetch robots.txt for mckinsey.com: Failed to perform, curl: (92) HTTP/2 stream 1 reset by server (error 0x2 INTERNAL_ERROR). See https://curl.se/libcurl/c/libcurl-errors.html first for more details.; assuming full disallow


[2026-11-11 06:08:05]:(verificador_portada) WARNING: Max retries exceeded for blocked request: https://imsinternet.com/


[2026-11-11 06:08:09]:(verificador_portada) ERROR: Failed after 1 attempts: Failed to perform, curl: (28) Connection timed out after 20000 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.


[2026-11-11 06:08:09]:(verificador_portada) WARNING: Failed to fetch robots.txt for conecel.com.ec: Failed to perform, curl: (28) Connection timed out after 20000 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.; assuming full disallow


... (3 outputs left out)

[2026-11-11 06:08:20]:(verificador_portada) WARNING: Failed to fetch robots.txt for mckinseycompanyinc.com: Failed to perform, curl: (28) Connection timed out after 20001 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.; assuming full disallow


[2026-11-11 06:08:25]:(verificador_portada) ERROR: Failed after 1 attempts: Failed to perform, curl: (28) Connection timed out after 20001 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.


[2026-11-11 06:08:25]:(verificador_portada) WARNING: Failed to fetch robots.txt for saludsa.com.ec: Failed to perform, curl: (28) Connection timed out after 20001 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.; assuming full disallow


6 portadas leídas de 13 dominios en 36 s


## 6. Puntaje de verificación

| Evidencia | Puntaje |
| --- | --- |
| El RUC de la empresa aparece en la página | 100 |
| Razón social (sin forma jurídica) dentro del texto | `partial_ratio` de rapidfuzz |
| Nombre comercial como palabra completa en título o nombre del sitio | 100 |

In [9]:
empresas = pendientes.set_index("expediente")

def puntuar(v: dict) -> dict:
    e = empresas.loc[v["expediente"]]
    base, comercial = partes_nombre(e["nombre"])
    texto = normalizar(v["texto"])
    cabecera = normalizar(v["title"] + " " + v["og_site_name"])
    ruc = isinstance(e["ruc"], str) and e["ruc"] in re.sub(r"\D", "", v["texto"])
    p_base = fuzz.partial_ratio(base, texto) if len(base) >= 8 else 0
    p_comercial = 100 if comercial and re.search(rf"\b{re.escape(comercial)}\b", cabecera) else 0
    puntaje = 100 if ruc else max(p_base, p_comercial)
    evidencia = "ruc" if ruc else ("razon_social" if p_base >= p_comercial else "nombre_comercial")
    return {"expediente": v["expediente"], "url_oficial": v["url_final"], "url_fuente": v["origen"],
            "puntaje": round(puntaje), "evidencia": evidencia, "status": v["status"]}

resultados = pd.DataFrame([puntuar(v) for v in visitas if v["status"] == 200])
resultados.sort_values(["expediente", "puntaje"], ascending=[True, False]) if len(resultados) else resultados

,expediente,url_oficial,url_fuente,puntaje,evidencia,status
2,742,https://fabril.com/,dominio_derivado,0,razon_social,200
5,47845,https://www.claro.com.ec/personas/,peeringdb,53,razon_social,200
4,47972,https://www.tigo.com.ec/,peeringdb,0,razon_social,200
3,48038,https://www.saludsa.com/,dominio_derivado,58,razon_social,200
0,86945,https://www.primax.com/ec/,dominio_derivado,50,razon_social,200
1,86945,https://www.primax.com/ec/,dominio_derivado,50,razon_social,200


## 7. La mejor URL por empresa y guardado incremental

In [10]:
hoy = dt.date.today().isoformat()
mejores = (resultados.sort_values("puntaje", ascending=False).drop_duplicates("expediente")
           if len(resultados) else pd.DataFrame(columns=["expediente", "url_oficial", "url_fuente", "puntaje", "evidencia"]))
nuevo = pendientes[["expediente", "nombre", "grupo", "criterio_piloto"]].merge(mejores, on="expediente", how="left")
nuevo["url_verificada"] = "no_encontrada"
nuevo.loc[nuevo["puntaje"] >= UMBRAL, "url_verificada"] = "si"
nuevo.loc[nuevo["url_oficial"].notna() & (nuevo["puntaje"] < UMBRAL), "url_verificada"] = "revisar"
nuevo["fecha_verificacion"] = hoy
nuevo["verificado_por"] = "notebook_04"

columnas = ["expediente", "nombre", "grupo", "criterio_piloto", "url_oficial", "url_fuente",
            "url_verificada", "puntaje", "evidencia", "fecha_verificacion", "verificado_por"]
conservar = previo[~previo["expediente"].isin(nuevo["expediente"])] if len(previo) else None
salida = pd.concat([conservar, nuevo[columnas]], ignore_index=True) if conservar is not None else nuevo[columnas]
salida.to_csv(SALIDA, index=False)
display(salida["url_verificada"].value_counts())
nuevo[["nombre", "url_oficial", "url_fuente", "puntaje", "evidencia", "url_verificada"]]

url_verificada
revisar          5
no_encontrada    5
si               4
Name: count, dtype: int64

,nombre,url_oficial,url_fuente,puntaje,evidencia,url_verificada
0,CONSORCIO ECUATORIANO DE TELECOMUNICACIONES S....,https://www.claro.com.ec/personas/,peeringdb,53.0,razon_social,revisar
1,OTECEL S.A.,https://www.tigo.com.ec/,peeringdb,0.0,razon_social,revisar
2,CORPORACION EL ROSADO S.A.,NaN,NaN,NaN,NaN,no_encontrada
3,PRIMAX COMERCIAL DEL ECUADOR SOCIEDAD ANONIMA,https://www.primax.com/ec/,dominio_derivado,50.0,razon_social,revisar
4,SOCIEDAD NACIONAL DE GALAPAGOS CA,NaN,NaN,NaN,NaN,no_encontrada
5,LA FABRIL S.A.,https://fabril.com/,dominio_derivado,0.0,razon_social,revisar
6,"MCKINSEY & COMPANY, INC. ECUADOR",NaN,NaN,NaN,NaN,no_encontrada
7,IMS-INTERNET MEDIA SERVICES ECUADOR S.A.,NaN,NaN,NaN,NaN,no_encontrada
8,SALUDSA SISTEMA DE MEDICINA PRE-PAGADA DEL ECU...,https://www.saludsa.com/,dominio_derivado,58.0,razon_social,revisar
9,BMI IGUALAS MEDICAS DEL ECUADOR S.A.,NaN,NaN,NaN,NaN,no_encontrada


## Revisión a mano

Las filas `revisar` y `no_encontrada` se corrigen directamente en `plata/urls_piloto.csv`: poner la URL, `url_fuente = manual`, `url_verificada = manual` y `verificado_por` con su nombre. Las corridas siguientes no las vuelven a tocar.